# ToolNode

"Tool Node ek aisa node hota hai jo jab AI ko external help chahiye hoti hai, tab kisi tool (API, calculator, database, web search, etc.) ko call karta hai aur uska result wapas AI ko de deta hai."

# Layman Example 📚

Socho AI ek student hai.

Tum puchte ho:

"Delhi ka current weather kya hai?"

AI khud weather yaad nahi rakh sakta.

To AI kya karega?

Woh apne dost (Weather Tool) ko phone karega.

You
 │
 ▼
AI
 │
 ▼
"Ek minute, weather check karta hoon."
 │
 ▼
Weather Tool
 │
 ▼
Returns: 32°C
 │
 ▼
AI
 │
 ▼
"Delhi ka weather 32°C hai."

Yahan Weather Tool ko call karne wala Tool Node hai.

In [ ]:
# LangGraph Flow
User
 │
 ▼
LLM Node
 │
 ▼
Need Tool?
 │
 ├── No ─────► Answer
 │
 └── Yes
        │
        ▼
    Tool Node
        │
        ▼
      Tool
        │
        ▼
     Result
        │
        ▼
     LLM Node
        │
        ▼
     Final Answer

# "A Tool Node in LangGraph is a node responsible for executing external tools whenever the LLM requires additional information or functionality. It calls the appropriate tool, receives the result, and passes it back to the LLM so it can generate the final response."

# Tool Condition

"Tool Condition ek decision point hota hai jo check karta hai ki LLM ko tool use karna hai ya nahi."

# "Tool Condition decide karta hai: Tool call hoga ya direct answer diya jayega."



# Code Example
from langgraph.prebuilt import tools_condition

builder.add_conditional_edges(

    "chatbot",

    tools_condition
)

What does tools_condition do?

It checks the LLM's output:

If the LLM requested a tool call → go to the Tool Node.

Otherwise → finish the graph (or follow the "no tool" path).

 # Code

In [6]:
# Install langgraph and langgraph-checkpoint explicitly
!pip install -U langgraph
!pip install -U langgraph-checkpoint

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.sqlite import SqliteSaver # Reverted to original import path
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from dotenv import load_dotenv
import sqlite3
import requests

ModuleNotFoundError: No module named 'langgraph.checkpoint.sqlite'

In [ ]:
load_dotenv()

In [ ]:
search_tool = DuckDuckGoSearchRun(region="us-en")


In [ ]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}

        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}


In [ ]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    r = requests.get(url)
    return r.json()

In [ ]:
tools = [search_tool, get_stock_price, calculator]
llm_with_tools = llm.bind_tools(tools)


In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [ ]:
conn = sqlite3.connect(database="chatbot.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)


In [ ]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")

graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
def retrieve_all_threads():
    all_threads = set()
    for checkpoint in checkpointer.list(None):
        all_threads.add(checkpoint.config["configurable"]["thread_id"])
    return list(all_threads)